# Fine-tuning from the ground up

In the Kaggle competition *Name That LLM*, each passage is an answer to a prompt, written by one of 18 LLMs. The task is to name the LLM. Here we fine-tune a pretrained language model to do it, building it up one idea at a time:

1. what fine-tuning is
2. train once, then test on prompts the model never saw
3. why one test isn't enough: cross-validation
4. learning rate × epochs
5. memorising vs generalising
6. more data

Every step is written out in this notebook. Section 2 trains live on a GPU (about a minute); the later sections load runs that were trained before the session on an A100, because each takes several minutes.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
from sklearn.metrics import balanced_accuracy_score
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          Trainer, TrainingArguments, set_seed)

transformers.logging.set_verbosity_error()  # hide the library's loading reports and warnings
transformers.logging.disable_progress_bar()

DATA = Path("data")        # the competition's train.csv
RESULTS = Path("oof")      # predictions and scores of runs trained before the session
BLUE, ORANGE, GREY, LIGHT, BAND = "#2a78d6", "#eb6834", "#898781", "#c3c2b7", "#e1e0d9"
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titlelocation": "left", "font.size": 10})

assert torch.cuda.is_available(), "This notebook needs a GPU"
print("GPU:", torch.cuda.get_device_name(0))

train = pd.read_csv(DATA / "train.csv")
LABELS = sorted(train.model.unique())   # the 18 LLMs, always in this order
print(f"{len(train):,} passages · {train.prompt_id.nunique()} prompts · {len(LABELS)} LLMs")

## 1. What is fine-tuning

**ModernBERT** has already learned how English works from a huge amount of text. To turn it into an LLM detector we:
- put a new **output layer** on top, with one score for each of the 18 LLMs;
- train the whole thing on our passages, so the new layer learns to read ModernBERT's features, and ModernBERT adjusts those features to this task.

In [ ]:
MODEL = "answerdotai/ModernBERT-base"
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=len(LABELS))
total = sum(p.numel() for p in model.parameters())
new = sum(p.numel() for name, p in model.named_parameters() if not name.startswith("model."))
print(f"ModernBERT + output layer: {total / 1e6:.0f}M weights.")
print(f"The new output layer: {new / 1e6:.1f}M of them ({new / total:.1%}), starting from random values.")
del model

ModernBERT reads **tokens**: whole words, pieces of words, or punctuation marks. It keeps capitals and punctuation, so style clues such as curly apostrophes and dashes are visible to it.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
ids = tokenizer("It’s not the stars — it’s the silence between them; that’s what remains.")["input_ids"]
print([tokenizer.decode([i]) for i in ids])
lengths = [len(x) for x in tokenizer(train.text.tolist())["input_ids"]]
print(f"Tokens per passage: average {np.mean(lengths):.0f}, longest {max(lengths)}")

## 2. Let's train it

The hidden Kaggle test set only contains prompts that are **not** in our training data. So we hold back 30 of our 150 prompts as a **test set**, train on the other 120, and score the model on the held-back passages. This is one **train/test split**.

We hold back every 5th prompt, with all of its passages, so no prompt is on both sides.

In [ ]:
prompts = sorted(train.prompt_id.unique())
test_prompts = prompts[4::5]                  # every 5th prompt: the 5th, 10th, … 150th
is_test = train.prompt_id.isin(test_prompts)
fit, test = train[~is_test], train[is_test]
print(f"train on: {fit.prompt_id.nunique()} prompts, {len(fit):,} passages")
print(f"test on:  {test.prompt_id.nunique()} prompts the model never sees, {len(test)} passages")

The model needs each passage as tokens, and each answer as a number from 0 to 17. A small `Dataset` class hands them to the trainer one at a time:

In [ ]:
LABEL_ID = {label: i for i, label in enumerate(LABELS)}

class Passages(torch.utils.data.Dataset):
    """Tokenised passages, plus the right answer for each (when we know it)."""

    def __init__(self, texts, labels=None):
        self.tokens = tokenizer(list(texts), truncation=True, max_length=512)
        self.labels = None if labels is None else [LABEL_ID[label] for label in labels]

    def __len__(self):
        return len(self.tokens["input_ids"])

    def __getitem__(self, i):
        item = {key: values[i] for key, values in self.tokens.items()}
        if self.labels is not None:
            item["labels"] = self.labels[i]
        return item

The settings:
- **learning rate** 5e-5: the size of each update. Fine-tuning takes small steps so the model keeps what it already knows.
- **epochs** 3: passes over the training passages.
- **batch size** 32: passages per update.
- **warm-up**: the learning rate starts at zero and rises over the first 10% of steps, then falls back to zero by the end.

The **loss** starts around 2.89 (guessing among 18) and should fall as the model learns.

In [ ]:
set_seed(0)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=len(LABELS))
args = TrainingArguments(
    output_dir="/tmp/finetune_workshop", report_to="none", save_strategy="no",
    learning_rate=5e-5, num_train_epochs=3, per_device_train_batch_size=32,
    warmup_steps=0.1,          # warm up over the first 10% of steps
    bf16=True,                 # faster 16-bit arithmetic on the GPU
    logging_steps=10,          # record the loss every 10 steps
)
trainer = Trainer(model=model, args=args, train_dataset=Passages(fit.text, fit.model),
                  data_collator=DataCollatorWithPadding(tokenizer))  # pads each batch to its longest passage
result = trainer.train()
print(f"Trained for {result.metrics['train_runtime']:.0f} seconds; average loss {result.training_loss:.2f}")

In [ ]:
losses = [(log["step"], log["loss"]) for log in trainer.state.log_history if "loss" in log]
plt.figure(figsize=(8, 3.4))
if losses:
    plt.plot(*zip(*losses), color=BLUE, lw=2)
plt.axhline(np.log(len(LABELS)), color=GREY, ls="--", lw=1)
plt.text(plt.xlim()[1], np.log(len(LABELS)) + 0.06, "guessing (2.89) ", color=GREY, ha="right")
plt.ylim(0, 3.3)
plt.xlabel("training step")
plt.ylabel("training loss")
plt.title("Training loss")
plt.show()

Now the test. `predict` gives the model's probabilities for each passage; its answer is the LLM with the highest one. **Balanced accuracy** is the share of each LLM's passages it names correctly, averaged over the 18 LLMs.

We also score it on a sample of passages it **trained on**. Keep that number in mind for section 5.

In [ ]:
def predict(texts):
    """The model's probabilities: one row per passage, one column per LLM."""
    logits = torch.tensor(trainer.predict(Passages(texts)).predictions).float()
    return torch.softmax(logits, dim=1).numpy()

def answers(probs):
    return np.array(LABELS)[probs.argmax(axis=1)]

probs = predict(test.text)
score = balanced_accuracy_score(test.model, answers(probs))
seen = fit.sample(len(test), random_state=0)  # passages it trained on
seen_score = balanced_accuracy_score(seen.model, answers(predict(seen.text)))
print(f"balanced accuracy on unseen prompts: {score:.3f}")
print(f"balanced accuracy on passages it trained on: {seen_score:.3f}")

A single fine-tuning run lands somewhere around **0.29–0.35** on these test passages, against 0.06 for guessing at random. Run it again and it lands somewhere else; section 3 shows why. It also gets only about half of the passages it *trained on* right: we come back to that in section 5.

Four of its test passages, with its top three guesses:

In [ ]:
for i in np.random.default_rng(0).choice(len(test), size=4, replace=False):
    top3 = probs[i].argsort()[::-1][:3]
    truth = test.model.iloc[i]
    print("✓" if LABELS[top3[0]] == truth else "✗", "written by", truth)
    print("   ", test.text.iloc[i][:220], "…")
    print("    guesses:", " · ".join(f"{LABELS[j]} {probs[i, j]:.0%}" for j in top3))
    print()

When it misses, the right answer is often second or third, behind a close version from the same lab. Telling close versions apart is the hard part.

## 3. Cross-validation

`GroupKFold` makes 5 splits like section 2's: each holds back a different 30 prompts, with all their passages, so every prompt is tested exactly once. Split 0 is section 2's split.

In [ ]:
from sklearn.model_selection import GroupKFold

splits = list(GroupKFold(n_splits=5).split(train, groups=train.prompt_id))  # keeps each prompt's passages together
assert set(train.prompt_id.iloc[splits[0][1]]) == set(test_prompts)        # split 0 is section 2's split

for k, (tr, te) in enumerate(splits):
    print(f"split {k}: train on {train.prompt_id.iloc[tr].nunique()} prompts, "
          f"test on {train.prompt_id.iloc[te].nunique()} different ones ({len(te)} passages)")

Before the session we trained section 2's exact recipe 15 times: 3 seeds × 5 splits. Each run's score on each split is saved in `oof/arch_matrix.csv` (run `B0`):

In [ ]:
runs = pd.read_csv(RESULTS / "arch_matrix.csv")
baseline = runs[runs.config == "B0"]                 # 3 seeds of section 2's recipe
per_split = baseline[[f"split{k}" for k in range(5)]]

plt.figure(figsize=(9, 3.8))
for k in range(5):
    plt.scatter([k] * len(baseline), per_split[f"split{k}"], color=BLUE, alpha=0.7,
                label="one seed, one split" if k == 0 else None)
plt.scatter([0], [score], s=180, facecolors="none", edgecolors="black", lw=2, label="our run above")
plt.axhline(per_split.to_numpy().mean(), color=GREY, ls="--", label=f"average of all 15: {per_split.to_numpy().mean():.3f}")
plt.xticks(range(5), [f"split {k}" for k in range(5)])
plt.ylabel("balanced accuracy on its unseen prompts")
plt.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3)
plt.title("The same recipe, 3 seeds × 5 splits: one split's score is noisy")
plt.show()
print("5-split average for each seed:", ", ".join(f"{s:.3f}" for s in baseline.score))

The same recipe, on the same data:
- **One split's score ranges from about 0.29 to 0.37,** depending on which 30 prompts landed in the test set and on the random seed.
- **Our run above is just one more draw** from that spread.
- **Averaging all 5 splits steadies it,** though the three seeds' averages still differ by up to 0.02.

So from here on, every score is the **average over all 5 splits**. This is **cross-validation**. A change has to beat the baseline by about 0.03, twice the seed-to-seed spread, before we believe it.

## 4. Learning rate × epochs

Two settings matter most in fine-tuning:
- **learning rate:** how big each update is. Too small and the model barely moves; too big and it forgets what it knew.
- **epochs:** how many passes over the training data.

We tried 2 learning rates × 2 epoch counts before the session, each averaged over the 5 splits. The grey band is the **noise zone**: the baseline ± twice its seed-to-seed spread. Only a result outside the band counts.

In [ ]:
RECIPES = {"B0": "lr 5e-5, 3 epochs (section 2's recipe)", "T1": "lr 5e-5, 6 epochs",
           "T2": "lr 1e-4, 3 epochs", "T3": "lr 1e-4, 6 epochs"}
reference, noise = baseline.score.mean(), baseline.score.std()

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.axvspan(reference - 2 * noise, reference + 2 * noise, color=BAND, label="noise zone")
for y, (config, label) in enumerate(reversed(RECIPES.items())):
    scores = runs[runs.config == config].score
    real_gain = scores.mean() > reference + 2 * noise
    ax.scatter(scores, [y] * len(scores), color=BLUE if real_gain else GREY, s=45, zorder=3)
    ax.text(scores.max() + 0.003, y, f"{scores.mean():.3f}", va="center")
ax.set_yticks(range(len(RECIPES)), list(reversed(RECIPES.values())))
ax.set_ylim(-0.6, len(RECIPES) - 0.4)
ax.set_xlim(reference - 2 * noise - 0.01, runs[runs.config.isin(RECIPES)].score.max() + 0.02)
ax.set_xlabel("balanced accuracy, average of 5 splits (one dot per seed)")
ax.legend(frameon=False, loc="upper right")
ax.set_title("Learning rate × epochs: blue = a real gain")
plt.show()

- Doubling the epochs (6 instead of 3) clears the noise zone; doubling the learning rate alone doesn't.
- **Both together (learning rate 1e-4, 6 epochs) reach about 0.37**, up from 0.32.

## 5. Memorising vs generalising

Every run also scored itself on a sample of passages it **trained on** (column `seen`). Comparing that with its score on **unseen** prompts shows whether a model is still learning or just memorising.

In [ ]:
means = runs[runs.config.isin(RECIPES)].groupby("config")[["seen", "score"]].mean()
plt.figure(figsize=(7, 4.3))
plt.scatter(means.seen, means.score, color=BLUE, s=55, zorder=3)
for config, row in means.iterrows():
    plt.annotate(RECIPES[config], (row.seen, row.score), xytext=(-8, 4), textcoords="offset points", ha="right")
plt.xlim(0, 1.05)
plt.xlabel("passages it trained on, named correctly (memorising)")
plt.ylabel("unseen prompts (generalising)")
plt.title("Memorising vs generalising")
plt.show()

Section 2's recipe gets only about 55% of its *own training passages* right. It isn't memorising too much: it's **under-trained**. Here, memorising more (moving right) went with doing better on unseen prompts (moving up).

Does that last? One run of 12 epochs at learning rate 1e-4, scored after every epoch on both, shows the whole story (saved as `arch_CP_s0.json`):

In [ ]:
saved_run = json.loads((RESULTS / "arch_CP_s0.json").read_text())
points = [(epoch, unseen, seen) for split in saved_run["epoch_scores"].values() for epoch, unseen, seen in split]
curve = pd.DataFrame(points, columns=["epoch", "unseen", "seen"]).groupby("epoch").mean()  # average of 5 splits

plt.figure(figsize=(8, 4))
plt.plot(curve.index, curve.seen, color=ORANGE, marker="o", label="passages it trained on (memorising)")
plt.plot(curve.index, curve.unseen, color=BLUE, marker="o", label="unseen prompts (generalising)")
plt.ylim(0, 1.05)
plt.xlabel("epoch")
plt.ylabel("balanced accuracy")
plt.legend(frameon=False, loc="center right")
plt.title("12 epochs at learning rate 1e-4, scored after every epoch")
plt.show()

- **By epoch 8 the model has memorised its training passages:** it names 100% of them correctly.
- **Unseen-prompt accuracy climbs fast for 4 epochs**, then creeps up to about 0.38 and flattens.
- **After memorising, extra epochs add little:** there's almost nothing left in the training data to learn from. To go further, we need **more data**.

## 6. More data

The **Open Pool** is a public Kaggle dataset of about 239,000 passages from the same 18 LLMs, generated and shared by other competitors (the rules allow it). We add 500 per model (9,000 in all) to every split's *training* data, picked so their punctuation and length match the official passages. We still *test* only on official passages.

A like-for-like comparison: the same model and learning rate (5e-5) as section 2, averaged over all 5 splits. With five times more data, it makes 2 passes instead of 3.

In [ ]:
def saved(name):
    """A saved model's predictions: one row per passage (in train.csv order), one column per LLM."""
    return pd.read_parquet(RESULTS / f"{name}.parquet")[LABELS].to_numpy()

def bars(scores, title):
    """One bar per model; the last (newest) in blue."""
    names = list(scores)
    fig, ax = plt.subplots(figsize=(9, 0.55 * len(names) + 1.2))
    ax.barh(names[::-1], [scores[n] for n in names[::-1]], color=[BLUE] + [LIGHT] * (len(names) - 1), height=0.6)
    for y, n in enumerate(names[::-1]):
        ax.text(scores[n] + 0.005, y, f"{scores[n]:.3f}", va="center")
    ax.axvline(1 / len(LABELS), color=GREY, ls="--", lw=1)  # guessing
    ax.set_xlim(0, 0.6)
    ax.set_xlabel("balanced accuracy on unseen prompts")
    ax.set_title(title)
    plt.show()

def cv_score(probs, which=range(5)):
    """Balanced accuracy on each split's test passages, averaged over the splits."""
    return np.mean([balanced_accuracy_score(train.model.iloc[splits[k][1]], answers(probs[splits[k][1]]))
                    for k in which])

bars({
    "ModernBERT · lr 5e-5, 3 epochs (section 2's recipe)": cv_score(saved("modernbert_official")),
    "ModernBERT · lr 1e-4, 6 epochs (section 4)": cv_score(saved("arch_T3_s0")),
    "ModernBERT · lr 5e-5 + 500 pool passages per model": cv_score(saved("modernbert_pool500")),
}, "Average over 5 splits")

Which LLMs gained, and how many passages changed?

In [ ]:
truth = train.model.to_numpy()
before, after = answers(saved("modernbert_official")), answers(saved("modernbert_pool500"))
hit_rate = pd.DataFrame({"official only": pd.Series(before == truth).groupby(truth).mean() * 100,
                         "+ 500 pool passages": pd.Series(after == truth).groupby(truth).mean() * 100}
                        ).sort_values("+ 500 pool passages")

fig, ax = plt.subplots(figsize=(9, 6.5))
y = np.arange(len(hit_rate))
ax.hlines(y, hit_rate["official only"], hit_rate["+ 500 pool passages"], color="#c3c2b7", lw=2)
ax.scatter(hit_rate["official only"], y, color=GREY, s=40, zorder=3, label="official passages only")
ax.scatter(hit_rate["+ 500 pool passages"], y, color=BLUE, s=40, zorder=3, label="+ 500 pool passages per model")
ax.set_yticks(y, hit_rate.index)
ax.set_xlim(0, 100)
ax.set_xlabel("% of the LLM's passages named correctly")
ax.legend(frameon=False, loc="lower right")
ax.set_title("Hit rate per LLM")
plt.show()
print(f"{((before != truth) & (after == truth)).sum()} passages fixed, {((before == truth) & (after != truth)).sum()} broken")

The same comparison at section 4's better learning rate (1e-4). To save time it ran on 2 of the 5 splits, so both bars are scored on those 2:

In [ ]:
bars({"ModernBERT · lr 1e-4, official passages only": cv_score(saved("arch_B8_s0"), which=[0, 1]),
      "ModernBERT · lr 1e-4 + 500 pool passages per model": cv_score(saved("arch_P500_s0"), which=[0, 1])},
     "Average over splits 0 and 1")

The same story at the better learning rate: 500 extra passages per model add about 0.05.

## Take-home messages

- **Test on prompts the model has never seen, and average over several splits.** The same recipe scored anywhere from about 0.29 to 0.37 on a single split.
- **Learning rate and epochs matter most:** training longer and faster took ModernBERT from 0.32 to 0.37.
- **Memorising isn't the enemy here, but it is a ceiling:** once the model has memorised its training passages, more epochs stop helping.
- **More data beats more tuning:** 500 extra passages per model took ModernBERT from 0.33 to 0.41, the biggest single jump.